Data Loading

In [232]:
import pandas as pd
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

information of the datasets


In [233]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [234]:
credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


Data Cleaning

In [235]:
movies.isnull().sum()

,0
budget,0
genres,0
homepage,3091
id,0
keywords,0
original_language,0
original_title,0
overview,3
popularity,0
production_companies,0


In [236]:
movies.dropna(subset=['overview'],inplace=True)
movies.isnull().sum()

,0
budget,0
genres,0
homepage,3088
id,0
keywords,0
original_language,0
original_title,0
overview,0
popularity,0
production_companies,0


In [237]:
credits.isnull().sum()

,0
movie_id,0
title,0
cast,0
crew,0


In [238]:
movies.duplicated().sum()

np.int64(0)

In [239]:
credits.duplicated().sum()

np.int64(0)

In [240]:
movies = movies.merge(credits, on='title')

Feature Selection

In [241]:
movies=movies[['title','crew','cast','genres','keywords','overview','popularity','id']]

Feature Extraction: Genres and Keywords

In [242]:
import ast

In [243]:
def convert(obj):
    L = []

    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L

In [244]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

Feature Extraction: Cast and Crew

In [247]:
def convert3(obj):
    L = []
    counter = 0

    for i in ast.literal_eval(obj):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [248]:
movies['cast'] = movies['cast'].apply(convert3)

In [249]:
movies['crew']=movies['crew'].apply(convert3)


Feature Engineering: Creating Movie Tagscreating tags

In [252]:
movies['tags']=movies['overview']+movies['genres'].apply(lambda x:' '.join(x))

In [253]:
movies['tags']=movies['tags']+movies['keywords'].apply(lambda x:' '.join(x))

In [254]:
movies['tags']=movies['tags']+movies['cast'].apply(lambda x:' '.join(x))

In [255]:
movies['tags']=movies['tags']+movies['crew'].apply(lambda x:' '.join(x))

In [257]:
movies['tag']=movies['tags'].apply(lambda x:x.lower())

Final Dataset Preparation

In [258]:
new_df=movies[['id','title','tags','popularity']]
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4806 entries, 0 to 4805
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          4806 non-null   int64  
 1   title       4806 non-null   object 
 2   tags        4806 non-null   object 
 3   popularity  4806 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 150.3+ KB


Text Vectorization

In [259]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')
vector=cv.fit_transform(new_df['tags']).toarray()
vector.shape

(4806, 5000)

Cosine Similarity

In [260]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vector)

In [261]:
similarity.shape

(4806, 4806)

Movie Recommendation System

In [262]:
def get_recommendations(movie):

    movie_lower = movie.lower()
    titles = new_df['title'].str.lower()

    if movie_lower not in titles.values:
        return "Movie not found. Please check the title."

    index = titles[titles == movie_lower].index[0]
    distances = similarity[index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    recommendations = []

    for i in movies_list:
        movie_title = new_df.iloc[i[0]]['title']
        score = round(i[1], 3)
        recommendations.append(
            f"{movie_title} → Similarity: {score}"
        )

    return "\n".join(recommendations)

In [263]:
movie_titles = new_df['title'].tolist()
movie_titles[:5]

['Avatar',
 "Pirates of the Caribbean: At World's End",
 'Spectre',
 'The Dark Knight Rises',
 'John Carter']

In [264]:
get_recommendations('Avatar')

'Moonraker → Similarity: 0.416\nAliens → Similarity: 0.414\nSpaceballs → Similarity: 0.411\nAlien → Similarity: 0.403\nAlien³ → Similarity: 0.383'

Gradio Interface

In [265]:
!pip install -q gradio

In [266]:
import gradio as gr

In [267]:
demo = gr.Interface(
    fn=get_recommendations,
    inputs=gr.Dropdown(
        choices=movie_titles,
        label="Select a Movie"
    ),
    outputs=gr.Textbox(
        label="Recommended Movies"
    ),
    title="🎬 Movie Recommendation System",
    description="Select a movie to get 5 similar movie recommendations."
)

In [268]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/08/30 10:22:44 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>